# MTA Subway Stations EDA

## Data Dictionary:
- https://data.ny.gov/Transportation/MTA-Subway-Stations/39hk-dx4f/about_data 

In [12]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

mta_data = pd.read_csv('../data/MTA_Subway_Stations_2025.csv', low_memory=False)

mta_data[150:175]

,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference
150,D13,151,151,IND,Concourse,145 St,M,False,B D,Subway,40.824783,-73.944216,Uptown,Downtown,0,0,0,NaN,POINT (-73.944216 40.824783)
151,A14,152,152,IND,8th Av - Fulton St,135 St,M,False,C B,Subway,40.817894,-73.947649,Uptown,Downtown,0,0,0,NaN,POINT (-73.947649 40.817894)
152,A15,153,153,IND,8th Av - Fulton St,125 St,M,False,A C B D,Subway,40.811109,-73.952343,Uptown,Downtown,1,1,1,NaN,POINT (-73.952343 40.811109)
153,A16,154,154,IND,8th Av - Fulton St,116 St,M,False,C B,Subway,40.805085,-73.954882,Uptown,Downtown,0,0,0,NaN,POINT (-73.954882 40.805085)
154,A17,155,155,IND,8th Av - Fulton St,Cathedral Pkwy (110 St),M,False,C B,Subway,40.800603,-73.958161,Uptown,Downtown,0,0,0,NaN,POINT (-73.958161 40.800603)
155,A18,156,156,IND,8th Av - Fulton St,103 St,M,False,C B,Subway,40.796092,-73.961454,Uptown,Downtown,0,0,0,NaN,POINT (-73.961454 40.796092)
156,A19,157,157,IND,8th Av - Fulton St,96 St,M,False,C B,Subway,40.791642,-73.964696,Uptown,Downtown,0,0,0,NaN,POINT (-73.964696 40.791642)
157,A20,158,158,IND,8th Av - Fulton St,86 St,M,False,C B,Subway,40.785868,-73.968916,Uptown,Downtown,0,0,0,NaN,POINT (-73.968916 40.785868)
158,A21,159,159,IND,8th Av - Fulton St,81 St-Museum of Natural History,M,False,C B,Subway,40.781433,-73.972143,Uptown,Downtown,0,0,0,NaN,POINT (-73.972143 40.781433)
159,A22,160,160,IND,8th Av - Fulton St,72 St,M,False,C B,Subway,40.775594,-73.976410,Uptown,Downtown,0,0,0,NaN,POINT (-73.97641 40.775594)


In [13]:
mta_data.shape

(496, 19)

In [14]:
mta_data['GTFS Stop ID'].is_unique

True

In [15]:
mta_data['Station ID'].nunique()

493

## Note:
- three (3) duplicate Station ID numbers exist:
- 151, 167, 461
- Are of no concern considering 'GTFS Stop ID's are unique *

In [16]:
dup_ids = mta_data[mta_data['Station ID'].duplicated(keep=False)]  # Get all occurrences of duplicates
dup_ids

,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference
149,A12,151,151,IND,8th Av - Fulton St,145 St,M,False,A C,Subway,40.824783,-73.944216,Uptown,Downtown,0,0,0,NaN,POINT (-73.944216 40.824783)
150,D13,151,151,IND,Concourse,145 St,M,False,B D,Subway,40.824783,-73.944216,Uptown,Downtown,0,0,0,NaN,POINT (-73.944216 40.824783)
166,A32,167,167,IND,8th Av - Fulton St,W 4 St-Wash Sq,M,True,A C E,Subway,40.732338,-74.000495,Uptown,Downtown,1,1,1,NaN,POINT (-74.000495 40.732338)
167,D20,167,167,IND,6th Av - Culver,W 4 St-Wash Sq,M,True,B D F M,Subway,40.732338,-74.000495,Uptown,Downtown,1,1,1,NaN,POINT (-74.000495 40.732338)
461,R09,461,461,BMT,Astoria,Queensboro Plaza,Q,False,N W,Elevated,40.750582,-73.940202,Astoria,Manhattan,1,1,1,NaN,POINT (-73.940202 40.750582)
462,718,461,461,IRT,Flushing,Queensboro Plaza,Q,False,7,Elevated,40.750582,-73.940202,Outbound,Manhattan,1,1,1,NaN,POINT (-73.940202 40.750582)


# Approach:
- Use Latitude and Longitude to group stations to their respective Zip Codes and Census Tracts (Community District requires an API or another library, handle this later)
- Zip code not already present; must input using us_zip_codes.csv

In [17]:
from scipy.spatial import KDTree

# Finding each MTA station's zip code using KDTree:
zips = pd.read_csv('../data/us_zip_codes.csv')
zips_ny = zips[zips['state_id'] == 'NY']
tree = KDTree(zips_ny[['lat', 'lng']].values)

distances, indices = tree.query(mta_data[['GTFS Latitude', 'GTFS Longitude']].values)
mta_data['zip_code'] = zips_ny.iloc[indices]['zip'].values

mta_data.head()

,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference,zip_code
0,R01,1,1,BMT,Astoria,Astoria-Ditmars Blvd,Q,False,N W,Elevated,40.775036,-73.912034,Last Stop,Manhattan,0,0,0,NaN,POINT (-73.912034 40.775036),11105
1,R03,2,2,BMT,Astoria,Astoria Blvd,Q,False,N W,Elevated,40.770258,-73.917843,Astoria,Manhattan,1,1,1,NaN,POINT (-73.917843 40.770258),11102
2,R04,3,3,BMT,Astoria,30 Av,Q,False,N W,Elevated,40.766779,-73.921479,Astoria,Manhattan,0,0,0,NaN,POINT (-73.921479 40.766779),11102
3,R05,4,4,BMT,Astoria,Broadway,Q,False,N W,Elevated,40.761820,-73.925508,Astoria,Manhattan,0,0,0,NaN,POINT (-73.925508 40.76182),11106
4,R06,5,5,BMT,Astoria,36 Av,Q,False,N W,Elevated,40.756804,-73.929575,Astoria,Manhattan,0,0,0,NaN,POINT (-73.929575 40.756804),11106


In [18]:
import censusgeocode as cg


# If code fails, try forcing urllib3 back to the version that still included the appengine module:
# Either run this in your terminal, or uncomment the line below and run it in your notebook:
# !pip install "urllib3<2"

def get_census_tract(lat, lon):
    try:
        # Note: Census API uses x=longitude, y=latitude
        result = cg.coordinates(x=lon, y=lat)
        
        if result and 'Census Tracts' in result:
            tract_data = result['Census Tracts'][0]
            #print("All data:", tract_data).keys()  # Uncomment to see all available keys
            return tract_data['BASENAME']
        return "No Tract Found"
    except Exception as e:
        return f"Error: {e}"

# Using row 149 as an example:
lat, lon = 40.824783, -73.944216
tract_id = get_census_tract(lat, lon)

print(f"Census Tract: {tract_id}")


Census Tract: 231


In [19]:
# takes about 8-9 minutes to run
mta_data['Census Tract'] = mta_data.apply(lambda row: get_census_tract(row['GTFS Latitude'], row['GTFS Longitude']), axis=1)

In [20]:
mta_data['Census Tract Whole'] = mta_data['Census Tract'].str.split('.').str[0]

In [21]:
mta_data['Census Tract Whole'].sort_values()

464       1
401     100
209    1008
318     101
11      101
30      101
227     101
276     102
195     106
199    1072
163     109
136    1098
335      11
24       11
174      11
58      110
224     112
59      112
351    1124
352    1124
350    1126
468     113
10      113
226     113
317     113
6       114
482     114
400     114
131    1142
132    1144
0       115
186    1152
187    1166
88     1168
87     1174
86     1178
70      118
85     1182
162     119
371     119
469     119
27      119
238     119
188    1196
185    1198
90     1198
89     1198
83       12
399     120
223     120
189    1202
239     121
483     122
370     123
80      124
9       125
1       125
225     125
429     125
474     126
398     128
339     129
329      13
414      13
20       13
173      13
21       13
328      13
277     131
316     131
485     132
486     132
161     133
484     134
36      136
8       137
28      137
7       143
156     143
157     143
159     143
29      143
158     143
79  

## Note:
- Census Tracts are stringified numbers (must go back to put school data in same format)
- Decimals appear to break up densely populated areas into smaller Census Tracts

In [22]:
mta_data[mta_data['Census Tract Whole'].str.startswith('7')]

,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference,zip_code,Census Tract,Census Tract Whole
12,R18,13,13,BMT,Broadway - Brighton,28 St,M,True,R W,Subway,40.745494,-73.988691,Uptown,Downtown,0,0,0,NaN,POINT (-73.988691 40.745494),10119,76,76
33,R40,34,34,BMT,4th Av,53 St,Bk,False,R,Subway,40.645069,-74.014034,Manhattan,Southbound,0,0,0,NaN,POINT (-74.014034 40.645069),11220,76,76
34,R41,35,35,BMT,4th Av,59 St,Bk,False,N R,Subway,40.641362,-74.017881,Manhattan,Southbound,1,1,1,NaN,POINT (-74.017881 40.641362),11220,74,74
42,D27,43,43,BMT,Broadway - Brighton,Parkside Av,Bk,False,Q,Open Cut,40.655292,-73.961495,Manhattan,Southbound,0,0,0,NaN,POINT (-73.961495 40.655292),11226,798.02,798
49,D34,50,50,BMT,Broadway - Brighton,Avenue M,Bk,False,Q,Open Cut,40.617618,-73.959399,Manhattan,Southbound,0,0,0,NaN,POINT (-73.959399 40.617618),11230,768,768
106,M23,107,107,BMT,Jamaica,Broad St,M,True,J Z,Subway,40.706476,-74.011056,Brooklyn,Last Stop,0,0,0,NaN,POINT (-74.011056 40.706476),10271,7,7
166,A32,167,167,IND,8th Av - Fulton St,W 4 St-Wash Sq,M,True,A C E,Subway,40.732338,-74.000495,Uptown,Downtown,1,1,1,NaN,POINT (-74.000495 40.732338),10014,71,71
167,D20,167,167,IND,6th Av - Culver,W 4 St-Wash Sq,M,True,B D F M,Subway,40.732338,-74.000495,Uptown,Downtown,1,1,1,NaN,POINT (-74.000495 40.732338),10014,71,71
237,F21,237,237,IND,6th Av - Culver,Carroll St,Bk,False,F G,Subway,40.680303,-73.995048,Northbound,Southbound,0,0,0,NaN,POINT (-73.995048 40.680303),11231,77,77
260,F07,260,260,IND,Queens Blvd,75 Av,Q,False,E F,Subway,40.718331,-73.837324,Jamaica,Manhattan,0,0,0,NaN,POINT (-73.837324 40.718331),11375,769.01,769


In [23]:
mta_data.head()

,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference,zip_code,Census Tract,Census Tract Whole
0,R01,1,1,BMT,Astoria,Astoria-Ditmars Blvd,Q,False,N W,Elevated,40.775036,-73.912034,Last Stop,Manhattan,0,0,0,NaN,POINT (-73.912034 40.775036),11105,115,115
1,R03,2,2,BMT,Astoria,Astoria Blvd,Q,False,N W,Elevated,40.770258,-73.917843,Astoria,Manhattan,1,1,1,NaN,POINT (-73.917843 40.770258),11102,125,125
2,R04,3,3,BMT,Astoria,30 Av,Q,False,N W,Elevated,40.766779,-73.921479,Astoria,Manhattan,0,0,0,NaN,POINT (-73.921479 40.766779),11102,63,63
3,R05,4,4,BMT,Astoria,Broadway,Q,False,N W,Elevated,40.761820,-73.925508,Astoria,Manhattan,0,0,0,NaN,POINT (-73.925508 40.76182),11106,59,59
4,R06,5,5,BMT,Astoria,36 Av,Q,False,N W,Elevated,40.756804,-73.929575,Astoria,Manhattan,0,0,0,NaN,POINT (-73.929575 40.756804),11106,53,53


In [27]:
mta_data_dropped = mta_data[['Daytime Routes', 'GTFS Stop ID', 'Station ID', 'Stop Name', 'Borough', 'GTFS Latitude', 'GTFS Longitude', 'Census Tract', 'Census Tract Whole', 'zip_code']]

In [28]:
mta_data_dropped.sample(10)

,Daytime Routes,GTFS Stop ID,Station ID,Stop Name,Borough,GTFS Latitude,GTFS Longitude,Census Tract,Census Tract Whole,zip_code
64,D,B18,65,79 St,Bk,40.613501,-74.000610,182,182,11228
122,L,L12,123,Grand St,Bk,40.711926,-73.940670,495,495,11211
437,3,302,437,145 St,M,40.820421,-73.936245,232,232,10039
306,1,116,306,125 St,M,40.815581,-73.958372,211,211,10027
448,7,702,448,Mets-Willets Point,Q,40.754622,-73.845625,383.02,383,11368
363,6,604,363,Westchester Sq-E Tremont Av,Bx,40.839892,-73.842952,200,200,10461
165,A C E,A31,166,14 St,M,40.740893,-74.001690,81,81,10011
385,4,410,385,176 St,Bx,40.848480,-73.911794,217,217,10453
171,E,E01,171,World Trade Center,M,40.712582,-74.009781,21,21,10279
158,C B,A21,159,81 St-Museum of Natural History,M,40.781433,-73.972143,143,143,10024


In [30]:
mta_data_dropped.to_csv('../cleaned_data/MTA_Subway_Stations_w_Tracts+Zip_2025.csv', index=False)